# E2: transfer to unseen exact architectures

This notebook runs the frozen archive-1–31 E2 ladder on either Google Colab or
Kaggle: grouped H0, topology-destroyed H0, parameter-matched pooled control,
high-level fusion, and ExtraTrees. The split unit is a canonical ordered
high-level architecture (operators, connectivity, tensor dimensions, kernels,
strides, padding, pooling, and batch normalization); synthesis knobs and labels
are excluded.

Most exact architecture groups are singletons. That is not leakage: it makes the
test a deliberately hard transfer-to-an-unseen-exact-architecture test. It does
mean the result is not evidence from repeated realizations of every architecture.
The audit reports singleton rates, and final uncertainty resamples whole
architecture IDs. A coarser topology ID is retained only for sensitivity
reporting—it is never substituted into the primary split.

**Colab restart:** reconnect, set the same `ACTIVE_EXPERIMENT` and `SEED`, then
rerun cells 1–6. Tensors are downloaded again to ephemeral storage; manifests,
logs, predictions, and atomic epoch checkpoints remain in Google Drive.

**Suggested order:** run all five experiments at seed 42 first. Do not reuse E1-H0
weights: E2 has a different split, so its H0 is trained from scratch. E1 and E2
remain comparable because they share the same archive-1–31 cohort, tensor
revision, targets, preprocessing, optimizer, loss, and effective batch size.


## 1 — Platform and persistent storage

On Colab, place `wa_high_level_archives1_32.pt` at
`MyDrive/hls-surrogate-lab/e2_structural_v1/inputs/` once. On Kaggle, attach it as
an unpacked dataset. No ZIP discovery or extraction is performed.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, re, shutil, subprocess, sys, time
import torch

IN_COLAB = (
    "COLAB_RELEASE_TAG" in os.environ
    or importlib.util.find_spec("google.colab") is not None
)
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
assert IN_COLAB or IN_KAGGLE, "Run this notebook in Google Colab or Kaggle"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    INPUT = Path("/content/drive/MyDrive")
    PERSIST_ROOT = INPUT / "hls-surrogate-lab/e2_structural_v1"
else:
    WORK = Path("/kaggle/working")
    INPUT = Path("/kaggle/input")
    PERSIST_ROOT = WORK / "e2_structural_v1"

# Edit only these controls between runs.
ACTIVE_EXPERIMENT = "h0"  # h0 | topology_destroyed | pooled | fusion | extra_trees
SEED = 42                 # screen all models at 42; later use 7 and 137 if retained
HIGH_LEVEL_CACHE_OVERRIDE = None
PERSIST_ROOT_OVERRIDE = None
if PERSIST_ROOT_OVERRIDE:
    PERSIST_ROOT = Path(PERSIST_ROOT_OVERRIDE)

REPO_URL = "https://github.com/brios-polimi/hls-surrogate-lab.git"
REPO_REF = "68829040d8c32f3218409fbdf78d302dcf141d30"
TENSOR_REPO_ID = "BrendanRios/wa-hls4ml-tensors-hierarchical"
TENSOR_REVISION = "1999fbea0187d4cac859f37dd20488fd63eb8860"
ARCHIVE_COUNT = 31
EXPECTED_LABELS_SHA256 = "1e0e3dc80edc71098e0fd9a05c4683b304e6574944928c804570e795412db226"
EXPECTED_VOCAB_SHA256 = "2a40117368c2536372ffdd0109198bb787227cb0ef9e7da9ac9c1b41d4f1b2bb"
EXPECTED_HIGH_LEVEL_SHA256 = "1fc6a8392ee583b25e8f8a5e16ffe81ed1e753a5391c3e89a843aa60e733b078"
EXPECTED_MANIFEST_SHA256 = "b73aa59f556ae43b855ec9824800a6df4391a27a82921c6debfecd2625f4d7a4"
EXPECTED_AUDIT_SHA256 = "2f7f25387db20625ba27bc89817fea6a2645006aeb354dc8c244f8c59907e349"
EXPECTED_SPLIT_SHA256 = "7d3fb85fd8664e5b2ea01b627e0f8401426098255a6c8c693f505e163160f2ca"
EXPECTED_SIZES = {"train": 10701, "validation": 2297, "test": 2295, "exemplar": 886}
KERNEL_TYPES = ["2layer", "3layer", "conv1d", "conv2d", "dense_latency", "dense_resource", "rule4ml"]
assert ACTIVE_EXPERIMENT in {"h0", "topology_destroyed", "pooled", "fusion", "extra_trees"}
assert SEED in {42, 7, 137}, "E2 permits only preregistered seeds 42, 7, and 137"

REPO_DIR = WORK / "hls-surrogate-lab-e2"
EPHEMERAL_HF = WORK / "wa_hls4ml_e2_hf_cache"
PROTOCOL_DIR = PERSIST_ROOT / "protocol"
RESULTS_DIR = PERSIST_ROOT / "results" / f"seed{SEED}"
CONFIG_DIR = PERSIST_ROOT / "configs" / f"seed{SEED}"
ANALYSIS_DIR = PERSIST_ROOT / "analysis" / f"seed{SEED}"
for path in (PERSIST_ROOT, PROTOCOL_DIR, RESULTS_DIR, CONFIG_DIR, ANALYSIS_DIR):
    path.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

os.environ["HF_HOME"] = str(EPHEMERAL_HF)
os.environ["HF_HUB_CACHE"] = str(EPHEMERAL_HF / "hub")
os.environ["HF_XET_CACHE"] = str(EPHEMERAL_HF / "xet")
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
print("Platform:", "Colab" if IN_COLAB else "Kaggle")
print("Persistent E2 root:", PERSIST_ROOT)
print("Active:", ACTIVE_EXPERIMENT, "seed", SEED)


## 2 — Install and reconstruct the pinned E2 code

The public repository is checked out at the same immutable commit used by E1-H0.
A narrow, hash-checked patch embedded in this notebook adds only the E2 protocol,
controls, resume hardening, and analysis code. This preserves provenance even
before those changes are published upstream.


In [ ]:
%pip install -q torch-geometric "huggingface_hub[hf_xet]>=0.32" pyyaml pandas scikit-learn scipy matplotlib seaborn


In [ ]:
import base64, gzip

E2_PATCH_SHA256 = "ef7c7b654ec01b32ee76d5137f7ecdf46d63026c1c777bc3e167e3ae2cf85d70"
E2_PATCH_B64 = "..."
marker = REPO_DIR / ".e2_embedded_patch_sha256"
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REF], check=True)
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
assert commit == REPO_REF

if not marker.is_file() or marker.read_text().strip() != E2_PATCH_SHA256:
    status = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
    assert not status, f"Checkout has unexpected changes before E2 patch:\n{status}"
    patch_bytes = gzip.decompress(base64.b64decode(E2_PATCH_B64))
    assert hashlib.sha256(patch_bytes).hexdigest() == E2_PATCH_SHA256
    patch_path = WORK / "hls_surrogate_e2.patch"
    patch_path.write_bytes(patch_bytes)
    subprocess.run(["git", "-C", str(REPO_DIR), "apply", "--check", str(patch_path)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "apply", str(patch_path)], check=True)
    marker.write_text(E2_PATCH_SHA256 + "\n")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
test_env = os.environ.copy()
test_env["PYTHONPATH"] = str(REPO_DIR / "src")
for pattern in ("test_e0_protocols.py", "test_topology_augmentation.py"):
    completed = subprocess.run([
        sys.executable, "-m", "unittest", "discover", "-s", "tests",
        "-p", pattern, "-v",
    ], cwd=REPO_DIR, env=test_env, text=True, capture_output=True)
    print(completed.stdout, end="")
    print(completed.stderr, end="", file=sys.stderr)
    if completed.returncode:
        raise RuntimeError(f"E2 preflight failed for {pattern}")
sys.path.insert(0, str(REPO_DIR / "src"))
print("Base commit:", commit)
print("E2 patch SHA-256:", E2_PATCH_SHA256)


## 3 — Validate the high-level cache and freeze the structural split

Archive 32 may remain inside the cache, but the builder selects only archives
1–31 plus exemplar. The exact cache, label index, generated manifest, logical
membership hash, and audit are all pinned. The split optimizer may balance only
sample counts and DSP/BRAM presence flags—not target magnitudes.


In [ ]:
from huggingface_hub import hf_hub_download, login

# Never paste a token into notebook source. Add a secret named HF_TOKEN in the
# Colab key icon or Kaggle Add-ons > Secrets. This public dataset also works
# anonymously, although authentication improves Hub rate limits.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token and IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception as error:
        print("Colab HF_TOKEN secret unavailable; using anonymous Hub access:", type(error).__name__)
if not hf_token and IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as error:
        print("Kaggle HF_TOKEN secret unavailable; using anonymous Hub access:", type(error).__name__)
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

LABELS_PATH = Path(hf_hub_download(
    repo_id=TENSOR_REPO_ID, filename="labels.json", repo_type="dataset",
    revision=TENSOR_REVISION, token=hf_token, cache_dir=str(EPHEMERAL_HF),
))
VOCAB_DOWNLOAD = Path(hf_hub_download(
    repo_id=TENSOR_REPO_ID, filename="vocab.json", repo_type="dataset",
    revision=TENSOR_REVISION, token=hf_token, cache_dir=str(EPHEMERAL_HF),
))
assert file_sha256(LABELS_PATH) == EXPECTED_LABELS_SHA256
assert file_sha256(VOCAB_DOWNLOAD) == EXPECTED_VOCAB_SHA256

if HIGH_LEVEL_CACHE_OVERRIDE:
    cache_candidates = [Path(HIGH_LEVEL_CACHE_OVERRIDE)]
elif IN_COLAB:
    cache_candidates = [
        PERSIST_ROOT / "inputs/wa_high_level_archives1_32.pt",
        INPUT / "hls-surrogate-lab/wa_high_level_archives1_32.pt",
    ]
else:
    cache_candidates = sorted(INPUT.rglob("wa_high_level_archives1_32.pt"))
cache_candidates = [path for path in cache_candidates if path.is_file()]
assert cache_candidates, (
    "High-level cache not found. On Colab upload wa_high_level_archives1_32.pt "
    f"to {PERSIST_ROOT / 'inputs'}; on Kaggle attach its unpacked dataset."
)
matching = [path for path in cache_candidates if file_sha256(path) == EXPECTED_HIGH_LEVEL_SHA256]
assert matching, f"Found cache candidates, but none has the frozen SHA-256: {cache_candidates}"
assert len({path.resolve() for path in matching}) >= 1
HIGH_LEVEL_CACHE_PATH = matching[0]

MANIFEST_PATH = PROTOCOL_DIR / "architecture_grouped_structural_v1.json"
AUDIT_PATH = PROTOCOL_DIR / "audit.json"
build_command = [
    sys.executable, str(REPO_DIR / "scripts/build_e2_manifest.py"),
    "--tensor-index", str(LABELS_PATH),
    "--high-level-cache", str(HIGH_LEVEL_CACHE_PATH),
    "--output-dir", str(PROTOCOL_DIR), "--archives", "31", "--seed", "42",
]
subprocess.run(build_command, cwd=REPO_DIR, check=True)
assert file_sha256(MANIFEST_PATH) == EXPECTED_MANIFEST_SHA256
assert file_sha256(AUDIT_PATH) == EXPECTED_AUDIT_SHA256
manifest = json.loads(MANIFEST_PATH.read_text())
audit = json.loads(AUDIT_PATH.read_text())
assert {key: len(value) for key, value in manifest.items()} == EXPECTED_SIZES
assert audit["split_sha256"] == EXPECTED_SPLIT_SHA256
assert not audit["protocol"].get("group_leaks", [])
print("High-level cache:", HIGH_LEVEL_CACHE_PATH)
print("Frozen split:", EXPECTED_SIZES, EXPECTED_SPLIT_SHA256)
print("Architecture singleton audit:")
for family, row in audit["architecture_signatures"].items():
    print(f"  {family:15s} groups={row['groups']:4d} singleton_fraction={row['singleton_group_fraction']:.3f} max_group={row['maximum_group_size']}")


## 4 — Download the exact tensors to ephemeral storage

This intentionally re-downloads after a Colab runtime reset. Tensor data and the
Hugging Face cache stay outside Drive and outside `/kaggle/working`; only compact
experiment state is persistent. Kaggle datasets are already unpacked, so there is
no ZIP check or extraction step.


In [ ]:
from huggingface_hub import snapshot_download
all_paths = [row["tensor_path"] for rows in manifest.values() for row in rows]
assert len(all_paths) == len(set(all_paths)) == sum(EXPECTED_SIZES.values())
patterns = sorted({
    f"{Path(path).parts[0]}/{Path(path).parts[1]}/*.pt" for path in all_paths
})
if IN_KAGGLE:
    # This is the proven 04_04 Kaggle/Xet path.
    TENSOR_DIR = Path(snapshot_download(
        repo_id=TENSOR_REPO_ID, repo_type="dataset", revision=TENSOR_REVISION,
        token=hf_token, cache_dir=str(EPHEMERAL_HF), max_workers=8,
        allow_patterns=[*patterns, "labels.json", "vocab.json"],
    ))
else:
    # Colab's shared IP can exhaust Hub API metadata limits for a 16k-file
    # snapshot. We already know every immutable path from the signed manifest,
    # so fetch direct resolver URLs and make zero repository-list API calls.
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from urllib.parse import quote
    import random, requests, threading

    cache_snapshot = (
        EPHEMERAL_HF / "hub"
        / "datasets--BrendanRios--wa-hls4ml-tensors-hierarchical"
        / "snapshots" / TENSOR_REVISION
    )
    TENSOR_DIR = cache_snapshot
    TENSOR_DIR.mkdir(parents=True, exist_ok=True)
    for source, name in ((LABELS_PATH, "labels.json"), (VOCAB_DOWNLOAD, "vocab.json")):
        destination = TENSOR_DIR / name
        if not destination.is_file():
            shutil.copy2(source, destination, follow_symlinks=True)
    download_paths = [path for path in all_paths if not (TENSOR_DIR / path).is_file()]
    print(f"Direct resolver download: {len(download_paths)} missing of {len(all_paths)} tensors")
    local_state = threading.local()

    def download_from_resolver(relative_path):
        destination = TENSOR_DIR / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_suffix(destination.suffix + ".part")
        url = (
            f"https://huggingface.co/datasets/{TENSOR_REPO_ID}/resolve/"
            f"{TENSOR_REVISION}/{quote(relative_path, safe='/')}?download=true"
        )
        if not hasattr(local_state, "session"):
            local_state.session = requests.Session()
        headers = {"Authorization": f"Bearer {hf_token}"} if hf_token else {}
        for attempt in range(10):
            response = local_state.session.get(
                url, headers=headers, stream=True, timeout=(30, 300),
                allow_redirects=True,
            )
            if response.status_code == 429:
                delay = max(float(response.headers.get("Retry-After", 0)), min(60, 2 ** attempt))
                response.close()
                time.sleep(delay + random.random())
                continue
            response.raise_for_status()
            with temporary.open("wb") as handle:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        handle.write(chunk)
            temporary.replace(destination)
            return relative_path
        raise RuntimeError(f"Resolver remained rate-limited for {relative_path}")

    with ThreadPoolExecutor(max_workers=16) as pool:
        futures = [pool.submit(download_from_resolver, path) for path in download_paths]
        for completed, future in enumerate(as_completed(futures), 1):
            future.result()
            if completed % 250 == 0 or completed == len(futures):
                print(f"Resolver files: {completed}/{len(futures)}", flush=True)
VOCAB_PATH = TENSOR_DIR / "vocab.json"
assert file_sha256(VOCAB_PATH) == EXPECTED_VOCAB_SHA256
missing = [path for path in all_paths if not (TENSOR_DIR / path).is_file()]
assert not missing, f"Incomplete tensor snapshot; first missing: {missing[:5]}"
print("Ephemeral tensor snapshot:", TENSOR_DIR)
print("Validated tensors:", len(all_paths))


## 5 — Preflight the intervention and parameter control

The audit runs on real graphs from every family. It proves that node features,
membership/call relations, source lists, destination multisets, and cross-function
edges are unchanged, while intra-function instruction/def-use/block destinations
actually change. Both train and evaluation datasets receive the same deterministic
transform. The pooled model must be within 5% of H0's trainable parameter count.


In [ ]:
TOPOLOGY_AUDIT_PATH = PROTOCOL_DIR / "topology_transform_real_graph_audit.json"
subprocess.run([
    sys.executable, str(REPO_DIR / "scripts/audit_e2_topology_transform.py"),
    "--manifest", str(MANIFEST_PATH), "--tensor-dir", str(TENSOR_DIR),
    "--output", str(TOPOLOGY_AUDIT_PATH), "--seed", "42", "--per-family", "8",
], cwd=REPO_DIR, check=True)
topology_audit = json.loads(TOPOLOGY_AUDIT_PATH.read_text())
assert topology_audit["transform_version"] == "within_function_destination_v2"
assert topology_audit["changed_destinations"] > 0

from ll_hls4ml.data.vocab import load_vocab
from ll_hls4ml.models.registry import build
vocabulary, max_pos, _ = load_vocab(VOCAB_PATH)
y_means, y_stds = torch.zeros(6), torch.ones(6)
shared = dict(
    instruction_vocab_size=len(vocabulary), y_means=y_means, y_stds=y_stds,
    hidden_dim=64, num_layers=3, dropout=0.15,
    use_global_features=True, use_context=True, context_mode="core",
    split_heads=True, hurdle_heads=True, hurdle_prediction_mode="threshold",
)
h0_model = build("hierarchical", edge_pos_vocab_size=max_pos, **shared)
destroyed_model = build("hierarchical_topology_destroyed", edge_pos_vocab_size=max_pos, **shared)
pooled_model = build(
    "pooled_control", instruction_vocab_size=len(vocabulary),
    y_means=y_means, y_stds=y_stds, hidden_dim=58, num_layers=3,
    num_var_embed_layers=2, dropout=0.15, pool="multi", node_aggr="concat",
    use_global_features=True, use_context=True, context_mode="core",
    split_heads=True, hurdle_heads=True, hurdle_prediction_mode="threshold",
)
counts = {
    "h0": sum(p.numel() for p in h0_model.parameters()),
    "topology_destroyed": sum(p.numel() for p in destroyed_model.parameters()),
    "pooled": sum(p.numel() for p in pooled_model.parameters()),
}
counts["pooled_relative_difference"] = (counts["pooled"] - counts["h0"]) / counts["h0"]
assert counts["h0"] == counts["topology_destroyed"]
assert abs(counts["pooled_relative_difference"]) <= 0.05
(PROTOCOL_DIR / "parameter_control.json").write_text(json.dumps(counts, indent=2) + "\n")
print("Parameter control:", counts)
print("Real-graph changed fraction:", topology_audit["changed_fraction_of_eligible"])


## 6 — Run or resume one E2 experiment

Change `ACTIVE_EXPERIMENT` in cell 1 and rerun this cell. Neural runs checkpoint
atomically every epoch and persist optimizer, scheduler, mixed-precision scaler,
early-stopping history, and RNG state (exact CUDA RNG continuation on the
single-GPU Colab path). The newest valid checkpoint is selected automatically. A
run interrupted before its first checkpoint is moved
to a timestamped quarantine folder and restarted without deleting evidence.

Colab A100 uses BF16. Kaggle T4 uses FP16 with gradient scaling; two Kaggle GPUs
use DDP with per-device batch 8. Every platform keeps effective global batch 16.
Do not resume the same run across a different precision/world-size signature.


In [ ]:
import shlex

GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1 or ACTIVE_EXPERIMENT == "extra_trees", "Enable a GPU for neural E2 runs"
USE_DDP = bool(IN_KAGGLE and GPU_COUNT > 1 and ACTIVE_EXPERIMENT != "extra_trees")
WORLD_SIZE = GPU_COUNT if USE_DDP else 1
EFFECTIVE_GLOBAL_BATCH = 16
assert EFFECTIVE_GLOBAL_BATCH % WORLD_SIZE == 0
PRECISION = (
    "bf16" if GPU_COUNT and torch.cuda.is_bf16_supported()
    else "fp16" if GPU_COUNT else "float32"
)

experiment_names = {
    "h0": f"e2_h0_structural_seed{SEED}",
    "topology_destroyed": f"e2_topology_destroyed_structural_seed{SEED}",
    "pooled": f"e2_pooled_structural_seed{SEED}",
    "fusion": f"e2_fusion_structural_seed{SEED}",
    "extra_trees": f"e2_extra_trees_structural_seed{SEED}",
}
model_names = {
    "h0": "hierarchical", "topology_destroyed": "hierarchical_topology_destroyed",
    "pooled": "pooled_control", "fusion": "hierarchical_high_level_fusion",
}
EXPERIMENT_NAME = experiment_names[ACTIVE_EXPERIMENT]
RUN_DIR = RESULTS_DIR / EXPERIMENT_NAME

common_config = {
    "experiment_name": EXPERIMENT_NAME,
    "model": model_names.get(ACTIVE_EXPERIMENT),
    "tensor_dir": str(TENSOR_DIR), "tensor_source_revision": TENSOR_REVISION,
    "vocab_path": str(VOCAB_PATH), "split_manifest_path": str(MANIFEST_PATH),
    "require_complete_split_manifest": True,
    "results_dir": str(RESULTS_DIR),
    "checkpoint_dir": str(RUN_DIR / "checkpoints"),
    "kernel_types": KERNEL_TYPES, "archive_count": 31,
    "study_id": "e2_architecture_grouped_structural_v1",
    "protocol_id": "architecture_grouped_structural_v1",
    "architecture_signature_version": "wa_hls4ml_ordered_layer_structure_v1",
    "topology_signature_version": "wa_hls4ml_ordered_layer_dag_v1",
    "seed": SEED, "family_balanced_sampling": False,
    "distributed_world_size": WORLD_SIZE,
    "effective_global_batch": EFFECTIVE_GLOBAL_BATCH,
    "batch_size": EFFECTIVE_GLOBAL_BATCH // WORLD_SIZE,
    "num_workers": 2, "worker_tmpdir": "/tmp", "pin_memory": True,
    "prefetch_factor": 2, "thread_prefetch": False,
    "precision": PRECISION, "epochs": 400, "patience": 20,
    "learning_rate": 1e-3, "weight_decay": 1e-4,
    "num_layers": 3, "dropout": 0.15,
    "use_global_features": True, "use_context": True, "context_mode": "core",
    "split_heads": True, "hurdle_heads": True,
    "hurdle_prediction_mode": "threshold", "loss": "log_huber_hurdle",
    "log_huber_delta": 0.35, "hurdle_classification_weight": 0.25,
    "early_stopping_metric": "smape", "gradient_clip_norm": 1.0,
    "lr_scheduler_patience": 8, "lr_scheduler_factor": 0.5,
    "min_learning_rate": 1e-6, "checkpoint_interval": 1, "verbose": 2,
}
if ACTIVE_EXPERIMENT in {"h0", "topology_destroyed", "fusion"}:
    common_config.update(hidden_dim=64, heads=1)
elif ACTIVE_EXPERIMENT == "pooled":
    common_config.update(hidden_dim=58, num_var_embed_layers=2, pool="multi", node_aggr="concat")
if ACTIVE_EXPERIMENT == "topology_destroyed":
    common_config.update(
        graph_transform="permute_destinations_within_function_v2",
        corruption_seed=42,
        topology_transform_version="within_function_destination_v2",
    )
if ACTIVE_EXPERIMENT == "fusion":
    common_config.update(
        high_level_cache=str(HIGH_LEVEL_CACHE_PATH), high_level_encoder="gatv2",
        high_level_cache_sha256=EXPECTED_HIGH_LEVEL_SHA256,
    )

def resume_signature(config):
    excluded = {"tensor_dir", "vocab_path", "results_dir", "checkpoint_dir", "split_manifest_path"}
    return {
        "base_commit": REPO_REF, "e2_patch_sha256": E2_PATCH_SHA256,
        "tensor_revision": TENSOR_REVISION, "manifest_sha256": EXPECTED_MANIFEST_SHA256,
        "split_sha256": EXPECTED_SPLIT_SHA256,
        "config": {key: value for key, value in config.items() if key not in excluded},
    }

def import_kaggle_run(expected):
    if not IN_KAGGLE or RUN_DIR.exists():
        return
    matches = []
    for path in INPUT.rglob("notebook_resume_signature.json"):
        if path.parent.name == EXPERIMENT_NAME:
            try:
                if json.loads(path.read_text()) == expected:
                    matches.append(path.parent)
            except Exception:
                pass
    if matches:
        fingerprints = {file_sha256(path / "notebook_resume_signature.json") for path in matches}
        assert len(fingerprints) == 1, f"Conflicting compatible prior runs: {matches}"
        shutil.copytree(matches[0], RUN_DIR)
        print("Imported unpacked prior Kaggle run:", matches[0])

def valid_checkpoints(run_dir):
    candidates = list((run_dir / "checkpoints").glob("*.pt")) if run_dir.exists() else []
    valid = []
    for path in candidates:
        try:
            payload = torch.load(path, map_location="cpu", weights_only=True)
            valid.append((int(payload["epoch"]), path))
        except Exception as error:
            print("Ignoring invalid checkpoint:", path, type(error).__name__)
    return sorted(valid, reverse=True)

def run_command(command, log_path):
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(REPO_DIR / "src")
    environment["LL_HLS4ML_TQDM"] = "0"
    environment["MPLCONFIGDIR"] = str(WORK / "matplotlib")
    print("$", shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", buffering=1) as log:
        process = subprocess.Popen(
            command, cwd=REPO_DIR, env=environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
            return process.wait()
        except KeyboardInterrupt:
            process.send_signal(2)
            process.wait(timeout=120)
            raise

if ACTIVE_EXPERIMENT == "extra_trees":
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    feature_cache = PERSIST_ROOT / "feature_cache/e2_core_context.pkl"
    command = [
        sys.executable, str(REPO_DIR / "scripts/run_e2_extratrees.py"),
        "--manifest", str(MANIFEST_PATH), "--tensor-dir", str(TENSOR_DIR),
        "--vocab", str(VOCAB_PATH), "--output-dir", str(RUN_DIR),
        "--feature-cache", str(feature_cache), "--seed", str(SEED),
    ]
    assert run_command(command, RUN_DIR / "run.log") == 0
else:
    expected_signature = resume_signature(common_config)
    import_kaggle_run(expected_signature)
    signature_path = CONFIG_DIR / f"{EXPERIMENT_NAME}_resume_signature.json"
    result_signature_path = RUN_DIR / "notebook_resume_signature.json"
    for existing_signature in (signature_path, result_signature_path):
        if existing_signature.is_file():
            assert json.loads(existing_signature.read_text()) == expected_signature, (
            "Existing run has a different code/data/training signature"
            )
    checkpoints = valid_checkpoints(RUN_DIR)
    completed = RUN_DIR / "summary.json"
    if completed.is_file():
        print("Already complete:", RUN_DIR)
    else:
        if RUN_DIR.exists() and any(RUN_DIR.iterdir()) and not checkpoints:
            quarantine = RUN_DIR.with_name(RUN_DIR.name + f"_interrupted_before_checkpoint_{int(time.time())}")
            RUN_DIR.replace(quarantine)
            print("Preserved pre-checkpoint partial run at:", quarantine)
        RUN_DIR.mkdir(parents=True, exist_ok=True)
        signature_path.write_text(json.dumps(expected_signature, indent=2) + "\n")
        payload = dict(common_config)
        checkpoints = valid_checkpoints(RUN_DIR)
        if checkpoints:
            payload["resume_checkpoint_path"] = str(checkpoints[0][1])
            print("Resuming epoch", checkpoints[0][0], "from", checkpoints[0][1])
        config_path = CONFIG_DIR / f"{EXPERIMENT_NAME}.json"
        config_path.write_text(json.dumps(payload, indent=2) + "\n")
        train = [sys.executable, str(REPO_DIR / "scripts/train.py"), "--config", str(config_path)]
        if USE_DDP:
            train = [
                sys.executable, "-m", "torch.distributed.run", "--standalone",
                f"--nproc_per_node={WORLD_SIZE}", str(REPO_DIR / "scripts/train.py"),
                "--config", str(config_path),
            ]
        assert run_command(train, CONFIG_DIR / f"{EXPERIMENT_NAME}_training.log") == 0
        assert completed.is_file(), "Training exited without a completed summary"
    result_signature_path.write_text(json.dumps(expected_signature, indent=2) + "\n")

print("Run directory:", RUN_DIR)


## 7 — Package results and run available paired analysis

This cell is safe after any completed model. Once H0 and one or more controls are
present for the same seed, it validates exact test membership/targets and writes
10,000-replicate architecture-cluster bootstrap contrasts. Positive delta means
the candidate is worse than grouped H0. On Kaggle, the ZIP is only an export;
resume discovery still expects an attached, unpacked output dataset.


In [ ]:
prediction_paths = {}
for name, experiment in experiment_names.items():
    path = RESULTS_DIR / experiment / "predictions.csv"
    if path.is_file():
        prediction_paths[name] = path
if "h0" in prediction_paths and len(prediction_paths) > 1:
    command = [
        sys.executable, str(REPO_DIR / "scripts/analyze_e2.py"),
        "--manifest", str(MANIFEST_PATH), "--reference", "h0",
        "--output-dir", str(ANALYSIS_DIR), "--seed", str(SEED),
        "--replicates", "10000",
    ]
    for name, path in prediction_paths.items():
        command.extend(["--prediction", f"{name}={path}"])
    subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print("Paired analysis needs completed H0 plus at least one control; found:", sorted(prediction_paths))

if IN_KAGGLE:
    export = shutil.make_archive(
        str(WORK / f"hls_surrogate_e2_structural_seed{SEED}_results"),
        "zip", root_dir=RESULTS_DIR,
    )
    print("Kaggle export:", export)
else:
    print("Results are already persistent in Drive:", RESULTS_DIR)


## Interpretation guardrails

- The primary claim is performance on **unseen exact architecture signatures
  within known kernel families**. High singleton rates make this stringent but
  limit within-architecture replication.
- `topology_id` collapses dimensions and is reported as sensitivity context; it
  does not define the primary split.
- The topology-destroyed run is a retrained intervention, not an inference-time
  ablation. A positive destroyed-minus-H0 delta supports a need for adjacency
  beyond preserved node features, edge marginals, hierarchy, and call structure.
- Pooled H0 is parameter-matched within 5%. ExtraTrees sees deterministic graph
  summaries and synthesis context but no adjacency. Fusion tests whether the
  high-level representation adds information under the identical E2 membership.
- Seed 42 is the screening run. Promote conclusions only after the preregistered
  additional seeds are run for the models retained by the screening rule.
